# Guide intelligent du Musée · 04 ResNet (jalon `m3`)

Partir d'un ResNet-18 pré-entraîné sur ImageNet, n'entraîner que sa tête puis affiner son dernier bloc, et le comparer au réseau dense sur les critères des bornes : précision, vitesse sur CPU, taille.

Contexte du projet, données et découpage : voir `01_donnees.ipynb`.

## 1. Mise en place

In [ ]:
import math
import platform
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader

from commun import (IMAGENET_MEAN, IMAGENET_STD, ArtworkDataset, ResNetClassifier, fit, get_device, load_images,
                    load_metadata, normalization_stats, set_seed, split_indices)

set_seed()
DEVICE = get_device()
meta, CLASSES = load_metadata()
X_all, y_all = load_images(meta)
splits = split_indices(meta)
MEAN, STD = normalization_stats(X_all, splits["train"])

print(f"PyTorch {torch.__version__}, calculs sur : {DEVICE}")
print(f"{len(meta)} œuvres, {len(CLASSES)} styles ; " + ", ".join(f"{k} : {len(i)}" for k, i in splits.items()))

## 2. Tête seule

Recette de la séance 7 : un ResNet-18 pré-entraîné sur ImageNet, dont la dernière couche (1 000 classes) est remplacée par une nouvelle tête à 5 sorties, seule entraînée. Les entrées sont préparées comme au pré-entraînement : agrandies de 128 à 224 pixels, puis normalisées avec la moyenne et l'écart-type d'ImageNet, pas avec ceux du train.

Les parties gelées restent en mode évaluation pendant l'entraînement : leurs batch norms gardent les statistiques d'ImageNet au lieu de se recalculer sur nos batchs, sinon le backbone ne serait pas vraiment gelé.

### 2.1 Modèle et paramètres

In [ ]:
N_EPOCHS = 20
BATCH_SIZE = 64
LR_HEAD = 1e-3
LN_C = math.log(len(CLASSES))  # perte d'un modèle qui répond au hasard : ln 5

datasets = {k: ArtworkDataset(X_all, y_all, splits[k], IMAGENET_MEAN, IMAGENET_STD) for k in ("train", "val")}
val_loader = DataLoader(datasets["val"], batch_size=256)
loss_fn = nn.CrossEntropyLoss()

set_seed()
resnet = ResNetClassifier(len(CLASSES)).to(DEVICE)
n_params = sum(p.numel() for p in resnet.parameters())
n_trainable = sum(p.numel() for p in resnet.parameters() if p.requires_grad)

print(f"Nouvelle tête : {resnet.net.fc}")
print(f"Paramètres au total : {n_params:,}".replace(",", " "))
print(f"Paramètres entraînables : {n_trainable:,}".replace(",", " ") + f", soit {100 * n_trainable / n_params:.2f} %")

### 2.2 Entraînement de la tête

Mêmes splits et même boucle (`fit`) que le réseau dense. Adam avec un taux d'apprentissage de 1e-3 sur la tête seule, batch de 64, 20 époques, graine 0. Le test n'est pas utilisé.

In [ ]:
def summarize(history: dict[str, list[float]]) -> dict[str, float]:
    """Valeurs à la dernière époque et minimum de la perte de validation."""
    best = int(np.argmin(history["val_loss"]))
    return {"perte_train": history["train_loss"][-1], "perte_val": history["val_loss"][-1],
            "acc_train": history["train_acc"][-1], "acc_val": history["val_acc"][-1],
            "epoque_min_perte_val": best + 1, "min_perte_val": history["val_loss"][best]}


train_loader = DataLoader(datasets["train"], batch_size=BATCH_SIZE, shuffle=True)
optimizer = torch.optim.Adam(resnet.net.fc.parameters(), lr=LR_HEAD)
t0 = time.perf_counter()
head_history = fit(resnet, train_loader, val_loader, loss_fn, optimizer, N_EPOCHS, DEVICE)
epoch_s = (time.perf_counter() - t0) / N_EPOCHS
head = summarize(head_history)

print(f"Dernière époque : accuracy {head['acc_train']:.3f} en train, {head['acc_val']:.3f} en validation "
      f"(écart {head['acc_train'] - head['acc_val']:.3f}) ; perte {head['perte_train']:.3f} en train, "
      f"{head['perte_val']:.3f} en validation")
print(f"Minimum de la perte de validation à l'époque {head['epoque_min_perte_val']} ({head['min_perte_val']:.3f})")
print(f"Meilleure accuracy de validation : {max(head_history['val_acc']):.3f}, "
      f"à l'époque {int(np.argmax(head_history['val_acc'])) + 1}")
print(f"Durée d'une époque : {epoch_s:.1f} s, calculs sur {DEVICE} ({platform.platform()})")

### 2.3 Courbes de suivi

In [ ]:
# Palette de référence (créneaux 1 et 2), encre et grille discrètes, comme dans 03_diagnostic
COLORS = {"train": "#2a78d6", "val": "#eb6834"}
LABELS = {"train": "train", "val": "validation"}
INK, MUTED = "#52514e", "#898781"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.titlesize": 11, "font.size": 10,
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 1.5,
})


def plot_curves(ax, history: dict[str, list[float]], metric: str, chance: float) -> None:
    """Courbes train et validation d'une métrique ('loss' ou 'acc'), étiquetées en bout de ligne,
    avec en pointillés la valeur d'un modèle qui répond au hasard."""
    epochs = np.arange(1, len(history[f"train_{metric}"]) + 1)
    ends = {split: history[f"{split}_{metric}"][-1] for split in ("train", "val")}
    low, high = sorted(ends, key=ends.get)
    for split in ("train", "val"):
        ax.plot(epochs, history[f"{split}_{metric}"], color=COLORS[split], label=LABELS[split])
        nudge = -6 if split == low else 6  # écarte les deux étiquettes quand les courbes finissent proches
        ax.annotate(LABELS[split], (epochs[-1], ends[split]), xytext=(5, nudge), textcoords="offset points",
                    va="center", color=INK, fontsize=9)
    ax.axhline(chance, color=MUTED, linestyle="--", linewidth=1, label="réponse au hasard")
    ax.set_xlabel("époque")
    ax.set_xlim(1, epochs[-1] * 1.18)  # place pour les étiquettes en bout de ligne
    ax.set_xticks([1, *range(5, epochs[-1] + 1, 5)])


fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(11, 3.8))
plot_curves(ax_loss, head_history, "loss", LN_C)
ax_loss.set_ylabel("perte (entropie croisée)")
ax_loss.set_title("Perte")
plot_curves(ax_acc, head_history, "acc", 1 / len(CLASSES))
ax_acc.set_ylabel("accuracy")
ax_acc.set_title("Accuracy")
fig.legend(*ax_loss.get_legend_handles_labels(), loc="upper right", ncol=3, frameon=False)
fig.suptitle("ResNet-18, tête seule, par époque", x=0.01, ha="left", fontsize=12)
fig.tight_layout()
plt.show()

- **La tête seule apprend vite, avec très peu de paramètres** : en n'entraînant que 2 565 paramètres, l'accuracy de validation atteint 0,780 à la dernière époque, et 0,803 au mieux (époque 16). Les caractéristiques apprises sur ImageNet (contours, textures, motifs) séparent déjà bien les styles : une couche linéaire posée dessus suffit.
- **Peu de sur-apprentissage** : à la dernière époque, 0,863 en train contre 0,780 en validation, un écart de 0,083. La perte de validation est minimale à l'époque 17 (0,608) et reste presque plate ensuite : 20 époques suffisent.
- **C'est très au-dessus du réseau dense et de la régression logistique** de `03_diagnostic`. La section 4 les compare précisément, sur les modèles rechargés.
- **Coût** : environ 9,5 s par époque sur le GPU d'un Mac M1 (MPS), car chaque image traverse tout le backbone à chaque époque, même gelé.

### 2.4 Bilan

- **ResNet-18, tête seule : 0,780 d'accuracy de validation** à la dernière époque (graine 0), en n'entraînant que 2 565 des 11 179 077 paramètres (0,02 %).
- **Courbes saines** : pas de divergence, peu de sur-apprentissage (écart de 0,083), perte de validation minimale à l'époque 17.
- **Suite** (section 3) : dégeler `layer4` avec un taux d'apprentissage plus faible, pour adapter aux tableaux les motifs de haut niveau appris sur des photos.

## 3. Fine-tuning de `layer4`

- `layer4` dégelé, avec un taux d'apprentissage plus faible que celui de la tête.
- Le modèle fine-tuné doit battre la tête seule.

In [ ]:
# À compléter

## 4. Comparaison avec le réseau dense

- Accuracy de validation, temps d'inférence par image sur CPU et taille du modèle (contrainte des bornes).

In [ ]:
# À compléter